## Install Dependencies

In [1]:
# Uncomment if needed
!pip install -q torch transformers bitsandbytes accelerate langchain-huggingface ddgs PyPDF2 reportlab streamlit pyngrok langchain

import sys
print("Python version:", sys.version)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 69.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 78.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.6 MB/s eta 0:00:00:00:0100:01
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## Import Libraries

In [2]:
import os
import json
import torch
from PyPDF2 import PdfReader
try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph

## LLM Loader


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

_llm_instance = None
_tokenizer = None

def load_llm():
    global _llm_instance, _tokenizer
    if _llm_instance is not None:
        return _llm_instance

    print(f"Loading model: {MODEL_NAME} ...")

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    eos_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
        eos_ids.append(im_end_id)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        trust_remote_code=True,
        quantization_config=quant_config,
        torch_dtype=torch.bfloat16,
    )

    text_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=2050,
        max_length=4096,
        truncation=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.2,
        return_full_text=False,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id,
    )

    _tokenizer = tokenizer
    _llm_instance = HuggingFacePipeline(pipeline=text_pipeline)
    print("LLM Loaded")
    return _llm_instance


def llm_chat(llm, user_prompt, system_prompt=(
    "You are a precise assistant used inside an automated pipeline. "
    "Follow the user's formatting instructions exactly. When asked for "
    "JSON, output ONLY the JSON object and nothing else \u2014 no markdown "
    "fences, no explanation, no repeated copies of the answer."
)):
    """
    Formats the prompt with the model's real chat template before calling
    it. Calling an Instruct model with a bare f-string (no chat template,
    no system/user turns) is why the pipeline used to loop or repeat its
    own answer \u2014 the model never saw a proper "end of turn" marker.
    """
    if _tokenizer is None:
        load_llm()
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    formatted_prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    response = llm.invoke(formatted_prompt)
    if hasattr(response, "content"):
        response = response.content
    return str(response)


def extract_json(text):
    cleaned = text.replace("```json", "").replace("```", "").strip()
    decoder = json.JSONDecoder()
    for i, ch in enumerate(cleaned):
        if ch in "{[":
            try:
                obj, _ = decoder.raw_decode(cleaned[i:])
                return obj
            except json.JSONDecodeError:
                continue
    raise ValueError(f"No valid JSON found in LLM output:\n{text[:500]}")


# Initialize once for all agents
llm = load_llm()


Loading model: Qwen/Qwen2.5-7B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'eos_token_id', 'temperature', 'repetition_penalty', 'max_length', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Loaded


## Planner Agent

In [4]:
class PlannerAgent:
    """
    Planner Agent

    Responsibilities:
    -----------------
    1. Accept text or PDF input.
    2. Extract PDF text if needed.
    3. Build an execution plan.
    4. Return the plan as a Python dictionary.
    """

    def __init__(self, user_input=None, pdf_path=None, llm_instance=None, max_retries=2):
        self.user_input = user_input
        self.pdf_path = pdf_path
        self.llm = llm_instance or load_llm()
        self.max_retries = max_retries

    def read_pdf(self):
        if not self.pdf_path:
            raise ValueError("No PDF path provided.")
        if not os.path.exists(self.pdf_path):
            raise FileNotFoundError(f"{self.pdf_path} does not exist.")
        reader = PdfReader(self.pdf_path)
        pages = []
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                pages.append(page_text)
        return "\n".join(pages)

    def get_input(self):
        if self.pdf_path:
            return {
                "input_type": "pdf",
                "content": self.read_pdf()
            }
        if self.user_input:
            return {
                "input_type": "text",
                "content": self.user_input
            }
        raise ValueError("Please provide either text or a PDF file.")

    def build_prompt(self, data):
        json_schema = {
            "topic":"",
            "subtopics":[
                ""
            ],
            "keywords":[
                ""
            ],
            "input_type":"",
            "research_depth":"basic | medium | advanced",
            "agents":[
                {
                    "agent":"",
                    "task":"",
                    "input":"",
                    "output":"",
                    "priority":1
                }
            ]
        }
        MAX_CONTENT_CHARS = 6000
        content = data["content"]
        if len(content) > MAX_CONTENT_CHARS:
            content = content[:MAX_CONTENT_CHARS] + "\n...[truncated]"

        return f"""
You are the Planner Agent of a Multi-Agent AI Research Assistant.

Your ONLY responsibility is planning.

The user provides either:
- Plain text
- PDF document

Analyze the user's request.

Break the topic into multiple research subtopics.

Extract important keywords.

These subtopics will later be searched independently.

Available Agents:

1. RetrievalAgent
    - Search reliable information
    - Retrieve documents

2. AnalysisAgent
    - Analyze documents
    - Summarize findings

3. ReportAgent
    - Write a structured report

4. ExportAgent
    - Export the report as PDF

Input Type:
{data["input_type"]}

Return ONLY valid JSON — exactly one object matching this schema, nothing before or after it:

{json.dumps(json_schema, indent=4)}

User Content (this may be truncated if long; base your plan on what you see below):
{content}
"""

    def plan(self):
        data = self.get_input()
        prompt = self.build_prompt(data)

        last_error = None
        for attempt in range(1, self.max_retries + 1):
            response = llm_chat(self.llm, prompt)
            try:
                result = extract_json(response)
                result = self._normalize_plan(result)
                return result
            except ValueError as e:
                last_error = e
                if attempt < self.max_retries:
                    print(f"\u26a0\ufe0f Planner attempt {attempt} produced invalid JSON, retrying...")


        raise RuntimeError(
            f"PlannerAgent could not get valid JSON from the LLM after "
            f"{self.max_retries} attempts: {last_error}"
        )

    def _normalize_plan(self, result):
        """
        The LLM sometimes doesn't return the requested schema object:
        - [{"topic": ...}]  -> a single-item list wrapping the real dict
        - ["kw1", "kw2", ...] -> a flat list of keywords/terms (happens when
          the JSON-schema instructions got truncated out of the prompt for
          long documents, so the model falls back to literally "extracting
          keywords" as plain strings instead of the full object).
        Handle both instead of crashing the whole pipeline.
        """
        if isinstance(result, dict):
            return result

        if isinstance(result, list):
            if len(result) == 1 and isinstance(result[0], dict):
                return result[0]

            if result and all(isinstance(item, str) for item in result):
                print(
                    "\u26a0\ufe0f Planner returned a flat keyword list instead of the "
                    "expected schema object \u2014 building a fallback plan from it."
                )
                return {
                    "topic": result[0],
                    "subtopics": result[1:6],
                    "keywords": result,
                    "input_type": "pdf" if self.pdf_path else "text",
                    "research_depth": "basic",
                    "agents": [],
                }

            raise ValueError(
                f"Expected a single JSON object from the Planner, got a "
                f"list of {len(result)} items instead: {result!r}"
            )

        raise ValueError(
            f"Expected a JSON object from the Planner, got {type(result).__name__}: {result!r}"
        )

    def run(self):
        return self.plan()

print("PlannerAgent defined")

PlannerAgent defined


## Retrieval Agent


In [5]:
class RetrievalAgent:
    """
    Retrieval Agent
    Responsibilities
    ----------------
    1. Receive the execution plan from Planner Agent.
    2. Extract the research topic.
    3. Search the web.
    4. Return structured documents.
    """
    def __init__(self, plan, llm_instance=None):
        self.plan = plan
        self.llm = llm_instance or load_llm()

    def get_topics(self):
        if isinstance(self.plan, str):
            self.plan = json.loads(self.plan)

        topic = self.plan.get("topic", "")
        subtopics = self.plan.get("subtopics", [])
        keywords = self.plan.get("keywords", [])

        queries = []
        if topic:
            queries.append(topic)
        queries.extend(subtopics)
        queries.extend(keywords)

        queries = list(dict.fromkeys(queries))[:5]
        return queries

    def web_search(self, topic, max_results=5):
        documents = []
        queries = [
            topic,
            f"{topic} survey",
            f"{topic} latest research",
            f"{topic} review",
            f"{topic} techniques"]
        with DDGS() as ddgs:
            seen = set()
            for q in queries:
                for r in ddgs.text(q, max_results=8):
                    if len(documents) >= 15:
                        return documents
                    url = r.get("href", "")
                    if url not in seen:
                        seen.add(url)
                        documents.append({
                            "title": r.get("title","")[:200],
                            "source": url,
                            "content": r.get("body","")[:800]
                        })
        return documents

    def summarize_documents(self, documents):
        if not documents:
            return []
        text = ""
        for doc in documents:
            text += f"""
        Title:
        {doc['title']}

        Content:
        {doc['content']}

        ------------------------
        """
        prompt = f"""
        You are the Retrieval Agent.

        Summarize the following retrieved search results.

        Keep only the important information.

        Return ONLY JSON — exactly one object, nothing before or after it.

        Schema:

        {{
            "documents":[
                {{
                    "title":"",
                    "source":"",
                    "summary":""
                }}
            ]
        }}

        Results:

        {text}
        """
        response = llm_chat(self.llm, prompt)
        try:
            parsed = extract_json(response)
            if isinstance(parsed, dict) and "documents" in parsed:
                return parsed["documents"]
            return parsed
        except ValueError:
            return documents

    def retrieve(self):
        queries = self.get_topics()
        documents = []

        for query in queries:
            if len(documents) >= 20:
                break
            print(f"Searching for: {query}")
            docs = self.web_search(query)
            documents.extend(docs)

        unique_docs = {}
        for doc in documents:
            source = doc.get("source")
            if source:
                unique_docs[source] = doc
        documents = list(unique_docs.values())

        print(f"Found {len(documents)} unique docs, summarizing...")
        if not documents:
            print("No documents found — downstream analysis will have nothing real to work with.")

        return self.summarize_documents(documents)

    def run(self):
        return self.retrieve()

print("RetrievalAgent defined")

RetrievalAgent defined


## Analysis Agent


In [6]:
class AnalysisAgent:
    """
    Analysis Agent

    Responsibilities:
    -----------------
    1. Receive documents from the Retrieval Agent.
    2. Analyze and summarize the collected information.
    3. Identify key findings.
    4. Return structured JSON.
    """

    def __init__(self, documents, llm_instance=None):
        self.documents = documents
        self.llm = llm_instance or load_llm()

    def format_documents(self):
        text = ""
        for i, doc in enumerate(self.documents, start=1):
            title = doc.get("title", "Untitled")
            source = doc.get("source", "Unknown")
            content = doc.get("content", doc.get("summary", ""))
            text += f"""
Document {i}

Title:
{title}

Source:
{source}

Content:
{content}

----------------------------------------
"""
        return text

    def build_prompt(self):
        docs = self.format_documents()

        return f"""
You are an expert Research Analysis Agent.

Your job is NOT to summarize.
Your job is to synthesize information from multiple sources.

Instructions:

1. Read ALL retrieved documents carefully.
2. Remove duplicate information.
3. Group similar ideas together.
4. Identify the major research themes.
5. Compare different approaches.
6. Highlight advantages and disadvantages.
7. Identify conflicting findings.
8. Mention important recent techniques when supported by the retrieved documents.
9. If some requested topic is missing from the documents, explicitly mention that instead of inventing information.
10. NEVER hallucinate or create unsupported facts.

Return ONLY valid JSON.

Schema:

{{
    "topic_summary":"",

    "research_themes":[
        {{
            "theme":"",
            "description":""
        }}
    ],

    "key_findings":[
        ""
    ],

    "important_points":[
        ""
    ],

    "comparison":[
        {{
            "method":"",
            "advantages":"",
            "limitations":""
        }}
    ],

    "conflicts":[
        ""
    ],

    "missing_topics":[
        ""
    ],

    "recommendations":[
        ""
    ],

    "future_work":[
        ""
    ],

    "sources":[
        ""
    ]
}}

Retrieved Documents:

{docs}

Also extract and preserve all source URLs in the "sources" field of the JSON.

Return ONLY valid JSON.
"""

    def analyze(self):
        if not self.documents:

            return {
            "topic_summary":"No supporting evidence was retrieved.",
        
            "research_themes":[],
            "key_findings":[],
            "comparison":[],
            "conflicts":[],
            "missing_topics":["No documents retrieved."],
            "recommendations":[
                "Run retrieval again with broader search queries."
            ],
            "future_work":[]
                    }

        prompt = self.build_prompt()
        response = llm_chat(self.llm, prompt)

        try:
            return extract_json(response)
        except ValueError:
            return {
                "error": "Invalid JSON returned by Analysis Agent.",
                "raw_output": response,
            }

    def run(self):
        return self.analyze()

print("AnalysisAgent defined")


AnalysisAgent defined


## Report Agent


In [7]:
class ReportAgent:
    """
    Report Agent

    Responsibilities
    ----------------
    1. Receive analyzed data.
    2. Generate a structured research report.
    3. Return the report in Markdown.
    """

    def __init__(self, analysis_result, llm_instance=None):
        self.analysis_result = analysis_result
        self.llm = llm_instance or load_llm()

    def build_prompt(self):

      if isinstance(self.analysis_result, dict):
          analysis = json.dumps(self.analysis_result, indent=2)
      else:
          analysis = self.analysis_result

      return f"""
  You are the Report Agent of a professional Multi-Agent AI Research System.

  Your responsibility is to transform the analysis results into a high-quality academic research report.

  Instructions:

  - Use professional Markdown formatting.
  - Write in a formal academic tone.
  - Do NOT invent facts.
  - Use ONLY the provided analysis.
  - If any section has missing information, explicitly mention that.
  - Expand important findings into well-written paragraphs.
  - Avoid repetition.
  - Keep the report clear and logically organized.

  The report MUST include:

  # Title

  ## Executive Summary

  - Brief overview
  - Main contributions
  - Overall conclusion

  ## Introduction

  - Background
  - Importance of the topic

  ## Research Themes

  Explain every identified research theme.

  ## Key Findings

  Expand every finding into a detailed paragraph.

  ## Comparison of Approaches

  Create a markdown table.

  | Method | Advantages | Limitations |

  ## Important Insights

  Summarize the most valuable insights.

  ## Conflicting Information

  Explain disagreements between sources if they exist.

  If there are no conflicts, explicitly state that.

  ## Missing Topics

  Mention topics requested by the user but not sufficiently covered by the retrieved documents.

  ## Recommendations

  Provide practical recommendations.

  ## Future Work

  Suggest future research directions.

  ## Conclusion

  Summarize the overall report.

  Analysis:

  {analysis}

  Return ONLY the report.
  """

    def generate_report(self):
        prompt = self.build_prompt()
        response = llm_chat(self.llm, prompt)
        return response.strip()

    def save_report(self, report, filename="reports/final_report.md"):
        os.makedirs("reports", exist_ok=True)
        with open(filename, "w", encoding="utf-8") as f:
            f.write(report)
        return filename

    def run(self):
        report = self.generate_report()
        file_path = self.save_report(report)
        return {
            "report": report,
            "file": file_path
        }

print("ReportAgent defined")


ReportAgent defined


## Export Agent


In [8]:
class ExportAgent:
    """
    Export Agent

    Responsibilities
    ----------------
    1. Receive the final report.
    2. Export it to PDF.
    """

    def __init__(self, report_text):
        self.report_text = report_text

    def export_pdf(self, output_path="reports/final_report.pdf"):
        os.makedirs("reports", exist_ok=True)
        doc = SimpleDocTemplate(output_path)
        styles = getSampleStyleSheet()
        story = []
        # Split report into paragraphs
        for line in self.report_text.split("\n"):
            line = line.strip()
            if line:
                # Escape minimal XML for reportlab
                safe_line = line.replace("<", "&lt;").replace(">", "&gt;")
                try:
                    story.append(Paragraph(safe_line, styles["BodyText"]))
                except Exception as e:
                    print(f"\u26a0\ufe0f Skipping malformed line in PDF export: {e}")
                    story.append(Paragraph(" ", styles["BodyText"]))
        doc.build(story)
        return output_path

    def run(self):
        pdf_file = self.export_pdf()
        return {
            "status": "success",
            "pdf_file": pdf_file
        }

print("ExportAgent defined")


ExportAgent defined


## Main Orchestrator


In [9]:
class AIResearchAssistant:

    def __init__(self, shared_llm=None):
        self.llm = shared_llm or load_llm()

    def run(self, user_input=None, pdf_path=None):

        # Planner
        print("🚀 Running Planner Agent...")
        planner = PlannerAgent(
            user_input=user_input,
            pdf_path=pdf_path,
            llm_instance=self.llm
        )
        execution_plan = planner.run()  # raises RuntimeError if the LLM never returns valid JSON
        print("✅ Planner done:", execution_plan.get("topic", "No topic"))

        # Retrieval
        print("\n🚀 Running Retrieval Agent...")
        retriever = RetrievalAgent(execution_plan, llm_instance=self.llm)
        retrieved_documents = retriever.run()
        print(f"✅ Retrieval done: {len(retrieved_documents)} docs")

        # Analysis
        print("\n🚀 Running Analysis Agent...")
        analysis = AnalysisAgent(retrieved_documents, llm_instance=self.llm)
        analysis_result = analysis.run()
        if "error" in analysis_result:
            print("\u26a0\ufe0f Analysis Agent returned invalid JSON \u2014 report will note this instead of inventing findings.")
        print("✅ Analysis done")

        # Report
        print("\n🚀 Running Report Agent...")
        report_agent = ReportAgent(analysis_result, llm_instance=self.llm)
        report_result = report_agent.run()
        print("✅ Report done")

        # Export
        print("\n🚀 Running Export Agent...")
        exporter = ExportAgent(report_result["report"])
        export_result = exporter.run()
        print(f"✅ Export done: {export_result['pdf_file']}")

        return {
            "planner": execution_plan,
            "retrieval": retrieved_documents,
            "analysis": analysis_result,
            "report": report_result,
            "export": export_result
        }

print("AIResearchAssistant orchestrator defined")


AIResearchAssistant orchestrator defined


## Ngrok

In [ ]:
import os
import shutil
import threading
import time
import uuid
import traceback
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse, FileResponse
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI()
llm = load_llm()
assistant = AIResearchAssistant(llm)

REPORTS_DIR = "/kaggle/working/reports_out"
os.makedirs(REPORTS_DIR, exist_ok=True)


JOBS = {}
JOBS_LOCK = threading.Lock()


def _set_job(job_id, **fields):
    with JOBS_LOCK:
        JOBS[job_id].update(fields)


@app.get("/health")
def health():
    return {"status": "ok"}


def _run_text_job(job_id: str, topic: str):
    try:
        result = assistant.run(user_input=topic)
        _set_job(job_id, status="done", result=build_response(result))
    except Exception as e:
        error_msg = traceback.format_exc()
        print(f"Error in text job {job_id}:\n{error_msg}")
        _set_job(job_id, status="error", error=str(e), trace=error_msg)


def _run_pdf_job(job_id: str, saved_path: str):
    try:
        result = assistant.run(pdf_path=saved_path)
        _set_job(job_id, status="done", result=build_response(result))
    except Exception as e:
        error_msg = traceback.format_exc()
        print(f"Error in pdf job {job_id}:\n{error_msg}")
        _set_job(job_id, status="error", error=str(e), trace=error_msg)


@app.post("/research/text")
def research_text(topic: str = Form(...)):
    job_id = uuid.uuid4().hex
    with JOBS_LOCK:
        JOBS[job_id] = {"status": "pending"}
    threading.Thread(target=_run_text_job, args=(job_id, topic), daemon=True).start()
    return {"job_id": job_id}


@app.post("/research/pdf")
async def research_pdf(file: UploadFile = File(...)):
    job_id = uuid.uuid4().hex
    print(f"Received PDF: {file.filename}")
    safe_name = f"input_upload_{job_id}.pdf"
    saved_path = os.path.join("/kaggle/working", safe_name)
    with open(saved_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    print(f"File saved to {saved_path}, starting background job {job_id}...")

    with JOBS_LOCK:
        JOBS[job_id] = {"status": "pending"}
    threading.Thread(target=_run_pdf_job, args=(job_id, saved_path), daemon=True).start()
    return {"job_id": job_id}


@app.get("/status/{job_id}")
def job_status(job_id: str):
    with JOBS_LOCK:
        job = JOBS.get(job_id)
    if job is None:
        return JSONResponse(status_code=404, content={"error": "Unknown job_id"})

    if job["status"] == "pending":
        return {"status": "pending"}
    elif job["status"] == "error":
        return JSONResponse(
            status_code=500,
            content={"status": "error", "error": job.get("error", "Unknown error"), "trace": job.get("trace", "")},
        )
    else:  # done
        return {"status": "done", **job["result"]}


@app.get("/download/{report_id}")
def download_report(report_id: str):
    """Stream the PDF instead of embedding it as base64 in a JSON blob.
    This avoids huge single-response payloads that were breaking the
    ngrok tunnel (ERR_NGROK_3004)."""
    path = os.path.join(REPORTS_DIR, f"{report_id}.pdf")
    if not os.path.exists(path):
        return JSONResponse(status_code=404, content={"error": "Report not found"})
    return FileResponse(path, media_type="application/pdf", filename="research_report.pdf")


def build_response(result):
    pdf_path = result["export"]["pdf_file"]
    report_id = uuid.uuid4().hex
    dest_path = os.path.join(REPORTS_DIR, f"{report_id}.pdf")
    shutil.copyfile(pdf_path, dest_path)
    return {
        "topic": result["planner"].get("topic", "Research Result"),
        "report_markdown": result["report"]["report"],
        "report_id": report_id,
        "pdf_filename": "research_report.pdf",
    }


NGROK_AUTH_TOKEN = "3Gn9I0anGgbT9198AW5hCTncWab_6cdfh3U5w64AZg4MW99R1"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

if not globals().get("_server_thread_started", False):
    def run_server():
        uvicorn.run(app, host="0.0.0.0", port=8000)
    threading.Thread(target=run_server, daemon=True).start()
    _server_thread_started = True
    time.sleep(2)
else:
    print("ℹ️ Server thread already running in this kernel, reusing it.")

public_url = ngrok.connect(8000)
print("\n" + "=" * 50)
print(f"🔗 PUBLIC NGROK URL: {public_url}")
print("=" * 50 + "\n")
print("✅ Server is running. Copy the URL above into your Streamlit app.")

while True:
    time.sleep(60)

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🔗 PUBLIC NGROK URL: NgrokTunnel: "https://abreast-aqua-unhappily.ngrok-free.dev" -> "http://localhost:8000"

✅ Server is running. Copy the URL above into your Streamlit app.
INFO:     156.195.114.149:0 - "GET /health HTTP/1.1" 200 OK
🚀 Running Planner Agent...
INFO:     156.195.114.149:0 - "POST /research/text HTTP/1.1" 200 OK


Both `max_new_tokens` (=2050) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b7bc9e HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/1b2ff40f04d44409b64bf0dbc2b

Both `max_new_tokens` (=2050) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Found 30 unique docs, summarizing...
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GE

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=2050) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Retrieval done: 6 docs

🚀 Running Analysis Agent...
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.19

Both `max_new_tokens` (=2050) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Analysis done

🚀 Running Report Agent...
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0 - "GET /status/6566b65bee1b42ae972edf803269aa17 HTTP/1.1" 200 OK
INFO:     156.195.114.149:0